# 04.00 · Ejecutar el paso 4: Preparatoria CCI

Con un solo notebook se procesan las sesiones de `data/raw/4. Preparatoria CCI/`: para cada PDF
de la agenda se lee la ficha, se cruza con los registros previos, se redactan alertas y
preguntas, y se genera `data/processed/4. Preparatoria CCI.xlsx` (`04.01` a `04.05`).

Funciona como el `03.00`: cada notebook corre en su propio kernel con el Python de `.venv`, los
notebooks originales no se modifican y el flujo se detiene en el primer fallo. Solo los
documentos nuevos o modificados llaman a Azure; el resto sale de las respuestas guardadas. Al
final se agrega un resumen a `reports/4. Preparatoria CCI_historico_ejecuciones.xlsx`.

**Para una sesión nueva**: se crea la carpeta con su fecha (por ejemplo,
`data/raw/4. Preparatoria CCI/13 de noviembre de 2026/`), se ponen allí los PDF numerados de la
agenda y se ejecuta este notebook.

**Antes de ejecutar, se cierran en Positron los notebooks del paso 4.**

### Parámetros

`notebooks_a_ejecutar` lleva las rutas dentro de `notebooks/`, en el orden del flujo. Para
rehacer solo el Excel (por ejemplo, después de ajustar `ajustes_cruce` en el `04.03`), se dejan
las tres últimas líneas. `timeout_por_celda` está en segundos.

In [ ]:
notebooks_a_ejecutar = [
    "4. Preparatoria CCI/04.01_documentos.ipynb",
    "4. Preparatoria CCI/04.02_ficha_documentos.ipynb",
    "4. Preparatoria CCI/04.03_cruce_registros.ipynb",
    "4. Preparatoria CCI/04.04_alertas_preguntas.ipynb",
    "4. Preparatoria CCI/04.05_excel_preparatoria.ipynb",
]
timeout_por_celda = 3600

## 1. LOAD LIBRARIES

Las mismas del `03.00`: nbformat lee los notebooks, nbclient los ejecuta y jupyter_client fija
el Python del proyecto. Las rutas se toman de `caf_input_cci_preparatoria/config.py`.

In [ ]:
from datetime import datetime
import json
from pathlib import Path
import re
import sys
import tempfile
from time import perf_counter

from dotenv import dotenv_values
from IPython.display import display
from jupyter_client import KernelManager
from jupyter_client.kernelspec import KernelSpecManager
from nbclient import NotebookClient
import nbformat
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))  # para importar el paquete caf_input_cci_preparatoria

from caf_input_cci_preparatoria import config

## 2. LOAD DATA

Los valores del `.env` se leen sin imprimirlos, solo para ocultarlos si aparecieran en un
error. El `kernel.json` temporal le indica a Jupyter que use `.venv/bin/python`, como en el
`03.00`.

In [ ]:
directorio_notebooks = ROOT / "notebooks"
python_proyecto = ROOT / ".venv" / "bin" / "python"
assert python_proyecto.is_file(), "No se encuentra el intérprete del proyecto en .venv/bin/python"

archivo_historico = config.REPORTS / "4. Preparatoria CCI_historico_ejecuciones.xlsx"
archivo_fichas = config.INTERIM / "4. Preparatoria CCI" / "04.02_ficha_documentos" / "fichas.csv"
valores_privados = [valor for valor in dotenv_values(ROOT / ".env").values() if valor]

In [ ]:
nombre_kernel = "caf-proyecto"
carpeta_temporal = tempfile.TemporaryDirectory()
directorio_kernel = Path(carpeta_temporal.name) / nombre_kernel
directorio_kernel.mkdir()
especificacion_kernel = {
    "argv": [str(python_proyecto), "-m", "ipykernel_launcher", "-f", "{connection_file}"],
    "display_name": "Python del proyecto CAF",
    "language": "python",
}
(directorio_kernel / "kernel.json").write_text(json.dumps(especificacion_kernel), encoding="utf-8")
gestor_especificaciones = KernelSpecManager(
    kernel_dirs=[carpeta_temporal.name], ensure_native_kernel=False
)

## 3. PROCESS DATA

Cada notebook se ejecuta en un kernel nuevo con su propia carpeta como directorio de trabajo.
Del texto de salida se toma el uso de Azure (la línea `Uso de Azure: …` del `04.02` y del
`04.04`).

En la fila del `04.02` se anotan los documentos leídos y los nuevos (huella que no estaba en
`fichas.csv` antes de la corrida); en la del `04.05`, las filas del Excel.

In [ ]:
def ocultar(texto):
    for valor in valores_privados:
        texto = texto.replace(valor, "[OCULTO]")
    return texto


def texto_salidas(notebook):
    return "".join(
        salida.get("text", "") for celda in notebook.cells for salida in celda.get("outputs", [])
    )


def uso_de_azure(notebook):
    patron = (
        r"Uso de Azure: ([\d,]+) llamadas; ([\d,]+) tokens de entrada; ([\d,]+) tokens de salida"
    )
    encontrado = re.search(patron, texto_salidas(notebook))
    if encontrado is None:
        return {}
    valores = [int(numero.replace(",", "")) for numero in encontrado.groups()]
    return dict(zip(["Llamadas Azure", "Tokens entrada", "Tokens salida"], valores))


def fallo_interno(notebook):
    encontrado = re.search(r"^Falló .*$", texto_salidas(notebook), flags=re.MULTILINE)
    return encontrado[0] if encontrado else ""


def leer_fichas():
    if not archivo_fichas.exists():
        return pd.DataFrame(columns=["archivo", "huella"])
    return pd.read_csv(
        archivo_fichas, encoding="utf-8-sig", usecols=["archivo", "huella"], dtype=str
    )


def celda_con_error(notebook):
    for numero, celda in enumerate(notebook.cells, start=1):
        if any(salida.output_type == "error" for salida in celda.get("outputs", [])):
            return numero
    return None

In [ ]:
fecha_ejecucion = datetime.now().strftime("%Y-%m-%d %H:%M")
fichas_antes = leer_fichas()
resultados = []
for ruta in notebooks_a_ejecutar:
    nombre = Path(ruta).name
    carpeta = (directorio_notebooks / ruta).parent
    print(f"Ejecutando {ruta}...")
    notebook = nbformat.read(directorio_notebooks / ruta, as_version=4)
    for celda in notebook.cells:
        if celda.cell_type == "code":
            celda.outputs, celda.execution_count = [], None
    cliente = NotebookClient(
        notebook,
        km=KernelManager(kernel_name=nombre_kernel, kernel_spec_manager=gestor_especificaciones),
        timeout=timeout_por_celda,
        resources={"metadata": {"path": str(carpeta)}},
    )
    inicio = perf_counter()
    error = ""
    try:
        cliente.execute()
    except Exception as fallo:  # noqa: BLE001 - el fallo queda en el resumen y detiene el flujo
        tipo = getattr(fallo, "ename", type(fallo).__name__)
        error = ocultar(f"{tipo}: {getattr(fallo, 'evalue', fallo)}")
    error = error or ocultar(fallo_interno(notebook))
    resultados.append(
        {
            "Fecha y hora": fecha_ejecucion,
            "Notebook": nombre,
            "Estado": "Falló" if error else "Completado",
            "Duración (s)": round(perf_counter() - inicio, 1),
            **uso_de_azure(notebook),
            "Celda": celda_con_error(notebook) if error else None,
            "Error": error,
        }
    )
    if nombre.startswith("04.02") and not error:
        fichas = leer_fichas()
        nuevos = fichas.loc[~fichas["huella"].isin(fichas_antes["huella"]), "archivo"]
        resultados[-1].update(
            {
                "Documentos": len(fichas),
                "Documentos nuevos": len(nuevos),
                "Archivos nuevos": "; ".join(nuevos),
            }
        )
    if nombre.startswith("04.05") and not error:
        filas = re.search(r"\((\d+) documentos\)", texto_salidas(notebook))
        resultados[-1]["Filas Excel"] = int(filas[1]) if filas else None
    if error:
        print(f"Falló {nombre}: {error}")
        break
carpeta_temporal.cleanup()

## 4. SAVE DATA

Se muestra el resumen de esta corrida, una fila por notebook ejecutado, con el total de
llamadas y tokens de Azure. Las mismas filas se agregan a
`reports/4. Preparatoria CCI_historico_ejecuciones.xlsx`, que se crea la primera vez.

In [ ]:
columnas = [
    "Fecha y hora",
    "Notebook",
    "Estado",
    "Duración (s)",
    "Documentos",
    "Documentos nuevos",
    "Archivos nuevos",
    "Filas Excel",
    "Llamadas Azure",
    "Tokens entrada",
    "Tokens salida",
    "Error",
    "Celda",
]
columnas_enteras = [
    "Documentos",
    "Documentos nuevos",
    "Filas Excel",
    "Llamadas Azure",
    "Tokens entrada",
    "Tokens salida",
    "Celda",
]

resumen = pd.DataFrame(resultados).reindex(columns=columnas)
resumen[columnas_enteras] = resumen[columnas_enteras].astype("Int64")  # enteros, no 3.0
with pd.option_context("display.max_colwidth", None):
    display(resumen)
totales = resumen[["Llamadas Azure", "Tokens entrada", "Tokens salida"]].sum()
print(
    f"Azure en esta corrida: {totales['Llamadas Azure']} llamadas; "
    f"{totales['Tokens entrada']:,} tokens de entrada; "
    f"{totales['Tokens salida']:,} tokens de salida"
)

In [ ]:
anteriores = pd.read_excel(archivo_historico) if archivo_historico.exists() else None
historico = pd.concat([anteriores, resumen], ignore_index=True).reindex(columns=columnas)
historico[columnas_enteras] = historico[columnas_enteras].astype("Int64")
archivo_historico.parent.mkdir(parents=True, exist_ok=True)
historico.to_excel(archivo_historico, index=False)
print(f"{len(resumen)} filas agregadas a {archivo_historico.relative_to(ROOT)}")